In [ ]:
import math
from pathlib import Path
from textwrap import fill

from matplotlib import pyplot as plt
from tqdm.notebook import tqdm

import vascx.fundus.feature_sets  # Register the available feature sets.
from vascx.cli import make_examples
from vascx.fundus.features.derived import AVR, FeatureEvaluator
from vascx.fundus.loader import RetinaLoader
from vascx.shared.features import FeatureSet

# Plot explanations for selected images

Choose a feature set, a **resolved column name** from its biomarker table, and the image IDs you want to inspect. This notebook loads only those images and computes the selected measurement (and its dependencies). It reuses the feature set's exact measurement settings.

Run the notebook from the `notebooks/` directory. The defaults use prepared images bundled with the repository. For your own images, set `INPUT_DIR` to the output folder from `vascx run-models`; the loader also accepts the `rgb/`, `av/`, `vessels/`, `discs/` layout used by the example notebooks. Images must already have their segmentations and landmarks. Plots use the default, uncalibrated output units.

In [ ]:
# output folder from vascx run-models
INPUT_DIR = Path("../samples/fundus")
# Use the feature set that produced the biomarker column you want to inspect.
FEATURE_SET_NAME = "od_centered"
# Exact resolved column name, including the vessel type when present.
FEATURE_NAME = "cre_arteries"
# Image IDs are filenames without extensions; their order is preserved below.
IMAGE_IDS = ["DRIVE_22", "HRF_04_g", "HRF_07_dr"]
# Number of image panels per row; AVR uses paired artery/vein columns instead.
N_COLS = 3
# Destination for the complete explanation figure.
OUTPUT_PATH = Path("../samples/figures") / f"explanations_{FEATURE_SET_NAME}_{FEATURE_NAME}.png"

## Find the selected measurement

The lookup below uses the same name mapping as extraction. For example, `cre_arteries` selects arteries, while `cre_veins` selects veins. To see all available names for your feature set, run `list(feature_set.names().by_name)`.

Some columns belong to a measurement with multiple outputs: `od_area`, for example, uses the shared optic-disc plot. For AVR, the figure shows the underlying arterial and venous measurements alongside their ratio.

In [ ]:
feature_set = FeatureSet.get_by_name(FEATURE_SET_NAME)
if feature_set is None:
    raise ValueError(f"Unknown feature set: {FEATURE_SET_NAME!r}")
output = feature_set.resolve(FEATURE_NAME)
print(output.display_name)
print(output.feature.output_description(output.component, layer_name=output.target))

## Load the requested images

Only metadata is read when the loader is created. The image-loading loop uses your chosen IDs. A missing ID is reported before plotting so you can correct the list or input folder.

In [ ]:
if (INPUT_DIR / "preprocessed_rgb").is_dir():
    # Reuse the CLI's handling of bounds and fovea files for run-models outputs.
    examples = make_examples(INPUT_DIR)
    if not examples:
        raise ValueError(f"No prepared images found in {INPUT_DIR}")
    loader = RetinaLoader({example["id"]: example for example in examples})
else:
    loader = RetinaLoader.from_folder(INPUT_DIR)

image_ids = [str(image_id) for image_id in IMAGE_IDS]
if not image_ids:
    raise ValueError("Choose at least one image ID.")
if len(set(image_ids)) != len(image_ids):
    raise ValueError("Image IDs must be unique.")
if not isinstance(N_COLS, int) or N_COLS < 1:
    raise ValueError("N_COLS must be a positive integer.")
missing_ids = [image_id for image_id in image_ids if image_id not in loader.items]
if missing_ids:
    raise ValueError(f"Image IDs not found in {INPUT_DIR}: {missing_ids}")

retinas = []
for image_id in tqdm(image_ids, desc="Loading selected images"):
    retinas.append((image_id, loader.by_id(image_id)))

## Draw the explanations

Each panel is labelled with its image ID and vessel type. A component such as optic-disc area shares its feature's full explanation plot. For AVR, each row shows one image with arteries on the left and veins on the right. Missing measurements are shown as `N/A`.

In [ ]:
is_avr = isinstance(output.feature, AVR)
if is_avr:
    source = output.feature.source
    plot_features = [(source, "arteries"), (source, "veins")]
else:
    plot_features = [(output.feature, output.target)]

panels = [
    (image_id, retina, feature, target_name)
    for image_id, retina in retinas
    for feature, target_name in plot_features
]
ncols = 2 if is_avr else min(N_COLS, len(panels))
nrows = math.ceil(len(panels) / ncols)
fig, axes = plt.subplots(
    nrows, ncols, figsize=(5 * ncols, 5 * nrows),
    squeeze=False, layout="constrained",
)
evaluators = {image_id: FeatureEvaluator(retina) for image_id, retina in retinas}

for ax, (image_id, retina, feature, target_name) in tqdm(
    zip(axes.flat, panels), total=len(panels), desc="Drawing explanations",
):
    target = dict(retina.feature_targets(feature)).get(target_name)
    if target is None:
        raise ValueError(f"Image {image_id!r} has no {target_name!r} layer.")
    evaluator = evaluators[image_id]
    evaluator.validate([output.feature])
    value = evaluator.evaluate(feature, target_name)
    feature.plot(ax, target, computed_value=value, plot_caption=False)
    title = f"{image_id} — {target_name}"
    if is_avr:
        ratio = evaluator.evaluate(output.feature, "retina")
        title += f"\nAVR: {ratio:.3g}" if ratio is not None else "\nAVR: N/A"
    ax.set_title(title)
    ax.set_axis_off()

for ax in list(axes.flat)[len(panels):]:
    ax.set_axis_off()
fig.suptitle(f"{output.display_name} — {FEATURE_SET_NAME}")
caption = plot_features[0][0].plot_description()
if caption:
    caption = fill(" ".join(caption.split()), width=110)
    caption_height = 0.18 * (caption.count("\n") + 1) + 0.3
    fig.set_figheight(5 * nrows + caption_height)
    fig.text(0.02, 0.02, caption, ha="left", va="bottom", fontsize=9)
    fig.get_layout_engine().set(rect=(0, caption_height / fig.get_figheight(), 1, 1 - caption_height / fig.get_figheight()))
plt.show()

## Save the figure

The file contains the same figure displayed above. Change the selected measurement or image list and rerun the cells to create another explanation.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_PATH, dpi=150, bbox_inches="tight")
print(OUTPUT_PATH.resolve())